# Nemotron-3-Diarization benchmark on Colab
**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom.
Untested in Colab; if an install cell fails, use the exact install line from the model card (https://huggingface.co/nvidia/Nemotron-3-Diarization).

## 1. Check the GPU

In [ ]:
!nvidia-smi | head -15
import torch; print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch the runtime to a GPU (T4)'

## 2. Upload `nemotron_bench.zip`

In [ ]:
from google.colab import files
import os, zipfile
up = files.upload()  # pick nemotron_bench.zip
zipfile.ZipFile(next(iter(up))).extractall('/content')
%cd /content/nemotron_bench
!ls -A

In [ ]:
# Install NeMo from GitHub main, on top of the deps you already have
!pip -q install --no-deps --force-reinstall "git+https://github.com/NVIDIA-NeMo/Speech.git@main"
!pip -q install "setuptools<82" "protobuf>=6.31.1,<7"
!pip show nemo-toolkit | head -3

## 3. Install (several minutes). If it asks to restart the runtime, do it, then re-run cell 2's `%cd` line.

In [ ]:
!apt-get -qq install -y ffmpeg libsndfile1
!pip -q install -U pip wheel Cython packaging "setuptools<82"
!pip -q install -r requirements.txt
!pip -q install --no-deps --force-reinstall "git+https://github.com/NVIDIA-NeMo/Speech.git@main"
!pip -q install "setuptools<82" "protobuf>=6.31.1,<7"

## 4. Hugging Face login (only needed if the model download is refused)

In [ ]:
from huggingface_hub import login
login()

## 5. Smoke check: NeMo imports and the model loads on GPU

In [ ]:
import sys; sys.path.insert(0, '/content/nemotron_bench')
from nemo.collections.asr.models import SortformerEncLabelModel
m = SortformerEncLabelModel.from_pretrained('nvidia/Nemotron-3-Diarization').to('cuda').eval()
print('model loaded OK'); del m
import torch; torch.cuda.empty_cache()

## 6. Start the app and a public tunnel

In [ ]:
import subprocess, time, re, os
os.chdir('/content/nemotron_bench')
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared', shell=True)
subprocess.Popen('nohup streamlit run app.py --server.port 8501 --server.headless true > st.log 2>&1 &', shell=True)
time.sleep(8)
subprocess.Popen('nohup ./cloudflared tunnel --url http://localhost:8501 > cf.log 2>&1 &', shell=True)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('cf.log').read())
    if m: url = m.group(0); break
print('OPEN THIS:', url or 'no URL yet, see cf.log')

Open the link above. **Anyone with the link can use the app**, so don't share it. If the page shows an error, check the logs below.

In [ ]:
!tail -20 /content/nemotron_bench/st.log

## 7. Save results to Google Drive (Colab wipes files when the session ends)
Re-run after each batch of runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nemotron_runs
!cp -r /content/nemotron_bench/static/runs/. /content/drive/MyDrive/nemotron_runs/
!ls /content/drive/MyDrive/nemotron_runs

## 8. Optional: download everything as a zip

In [ ]:
!cd /content/nemotron_bench/static && zip -qr /content/runs.zip runs
from google.colab import files; files.download('/content/runs.zip')

## Notes
- Big videos upload slowly through the tunnel. Extract audio first: `ffmpeg -i in.mp4 -vn -ac 1 -ar 16000 out.wav`.
- If the GPU runs out of memory, set the ASR model to `medium` in the app sidebar.
- The tunnel URL changes every time you re-run cell 6.